## Bird Data

In this notebook, we process data from the eBird Basic Dataset obtained from https://ebird.org/data/download. eBird is a semi-structured citizen science project, and each row of the data contains information about an instance of a bird observation and the process used for that instance. The information includes species observed, number of birds observed, the location of the observation, type of observation, and the amount of effort spent to make the observation. In "Analytical Guidelines to Increase the Value of Citizen Science Data: Using eBird Data to Estimate Species Occurence", the authors advocate imposing a more structured protocol on the dataset by filtering the data and including covariates that account for variations in the observation process.

For this project, we will not be filtering the data to impose a more structured protocol, but we will be including the covariates provided by eBird in our analysis. 

### Importing the necessary modules

In [4]:
import pandas as pd
import zipfile
import numpy as np

### Global variables

We will be restricting our attention to American goldfinch sightings (and non-sightings) in the are enclosed by the grid below.

In [7]:
lat_min = 38
lat_max = 45
long_min = -90
long_max = -83

### eBird basic data

Each row of the eBird basic data (EBD) corresponds to a single observation event. The rows contain the number of American goldfinch observed, the location of the observation, the time of the observation, and covariates the account for variations in the observation process.

In [10]:
with zipfile.ZipFile("Data/ebd_US_amegfi_201801_201912_smp_relJun-2026.zip") as myzip:
    with myzip.open("ebd_US_amegfi_201801_201912_smp_relJun-2026.txt",'r') as file:
        with open("Data/amegfi.txt", "w", encoding = "utf-8") as file2: 
            file.seek(0)
            line = file.readline().decode()
            file2.write(line)
            s_line = line.split(sep = '\t')
            latInd = s_line.index("LATITUDE")
            longInd = s_line.index("LONGITUDE")
            for _ in range(10000000):
                line = file.readline().decode(encoding = 'utf-8')
                if line == '':
                    print('End of file!')
                    break
                s_line = line.split(sep = '\t')
                lat = float(s_line[latInd])
                long = float(s_line[longInd])
                if (lat >= lat_min) and (lat < lat_max) and (long >= long_min) and (long < long_max):
                    file2.write(line)

End of file!


In [11]:
ebd = pd.read_table("Data/amegfi.txt", low_memory = False)
ebd = ebd[ebd["APPROVED"] == 1]


print(ebd.info())
print(ebd.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380178 entries, 0 to 380177
Data columns (total 53 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   GLOBAL UNIQUE IDENTIFIER    380178 non-null  object 
 1   LAST EDITED DATE            380178 non-null  object 
 2   TAXONOMIC ORDER             380178 non-null  int64  
 3   CATEGORY                    380178 non-null  object 
 4   TAXON CONCEPT ID            380178 non-null  object 
 5   COMMON NAME                 380178 non-null  object 
 6   SCIENTIFIC NAME             380178 non-null  object 
 7   SUBSPECIES COMMON NAME      0 non-null       float64
 8   SUBSPECIES SCIENTIFIC NAME  0 non-null       float64
 9   EXOTIC CODE                 0 non-null       float64
 10  OBSERVATION COUNT           380178 non-null  object 
 11  BREEDING CODE               11076 non-null   object 
 12  BREEDING CATEGORY           11076 non-null   object 
 13  BEHAVIOR CODE 

### Sampling event data

Sampling event data (SED) contains one row for each eBird checklist that was submitted during the specified time window at the specified regions. We will only consider rows that represent checklists for which observers indicated that all detected species were reported (rows where "ALL SPECIES REPORTED" has a value of 1). A record in the SED but no record of a species in the eBird Basic Dataset (EBD) indicates a count of zero individuals of that species.

In [14]:
with zipfile.ZipFile("Data/ebd_US_amegfi_201801_201912_smp_relJun-2026.zip") as myzip:
    with myzip.open("ebd_US_amegfi_201801_201912_smp_relJun-2026_sampling.txt",'r') as file:
        with open("Data/sampling_amegfi.txt", "w", encoding = "utf-8") as file2: 
            file.seek(0)
            line = file.readline().decode()
            file2.write(line)
            s_line = line.split(sep = '\t')
            latInd = s_line.index("LATITUDE")
            longInd = s_line.index("LONGITUDE")
            for _ in range(100000000):
                line = file.readline().decode(encoding = 'utf-8')
                if line == '':
                    print('End of file!')
                    break
                s_line = line.split(sep = '\t')
                lat = float(s_line[latInd])
                long = float(s_line[longInd])
                if (lat >= lat_min) and (lat < lat_max) and (long >= long_min) and (long < long_max):
                    file2.write(line)


End of file!


In [15]:
sed = pd.read_table("Data/sampling_amegfi.txt", low_memory = False)

In [16]:
print(sed.info())
print(sed.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1182557 entries, 0 to 1182556
Data columns (total 34 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   LAST EDITED DATE           1182557 non-null  object 
 1   COUNTRY                    1182557 non-null  object 
 2   COUNTRY CODE               1182557 non-null  object 
 3   STATE                      1182557 non-null  object 
 4   STATE CODE                 1182557 non-null  object 
 5   COUNTY                     1182557 non-null  object 
 6   COUNTY CODE                1182557 non-null  object 
 7   IBA CODE                   225667 non-null   object 
 8   BCR CODE                   1148647 non-null  float64
 9   USFWS CODE                 20590 non-null    object 
 10  ATLAS BLOCK                47352 non-null    object 
 11  LOCALITY                   1182557 non-null  object 
 12  LOCALITY ID                1182557 non-null  object 
 13  LOCALITY TYP

### Processing the data

To create data containing both presence and absence (or non-detection) information, we need to match the SAMPLING EVENT IDENTIFIER of the rows in ebd and sed. 

In [19]:
df = sed.merge(ebd, how = "left", on = "SAMPLING EVENT IDENTIFIER", suffixes = ("", "_x"))
df["OBSERVATION COUNT"] = df["OBSERVATION COUNT"].replace(np.nan, '0')

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1182557 entries, 0 to 1182556
Data columns (total 86 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   LAST EDITED DATE             1182557 non-null  object 
 1   COUNTRY                      1182557 non-null  object 
 2   COUNTRY CODE                 1182557 non-null  object 
 3   STATE                        1182557 non-null  object 
 4   STATE CODE                   1182557 non-null  object 
 5   COUNTY                       1182557 non-null  object 
 6   COUNTY CODE                  1182557 non-null  object 
 7   IBA CODE                     225667 non-null   object 
 8   BCR CODE                     1148647 non-null  float64
 9   USFWS CODE                   20590 non-null    object 
 10  ATLAS BLOCK                  47352 non-null    object 
 11  LOCALITY                     1182557 non-null  object 
 12  LOCALITY ID                  1182557 non-n

We have obtained the raw dataset containing American goldfinch presence and absence data in the years 2018 and 2019 at the region specified the latitude and logitude bounds. The function below cleans the raw dataset.

In [21]:
def processBirdData(df):
    
    cols = [
        "OBSERVATION COUNT",
        "IBA CODE",
        "BCR CODE",
        "USFWS CODE",
        "LATITUDE",
        "LONGITUDE",
        "OBSERVATION DATE",
        "TIME OBSERVATIONS STARTED",
        "OBSERVATION TYPE",
        "DURATION MINUTES",
        "EFFORT DISTANCE KM",
        "EFFORT AREA HA",
        "ALL SPECIES REPORTED",
        "GROUP IDENTIFIER"
    ]

    # remove unnecessary columns    
    df = df[cols]

    # remove rows that do not provide counts
    df = df[df["OBSERVATION COUNT"].str.isdigit() == True]

    # to make sure that our presence/absence data is as accuarate as possible, we have to restrict to complete checklists
    df = df[df["ALL SPECIES REPORTED"] == 1]
    df = df.drop(["ALL SPECIES REPORTED"], axis = 1)

    # convert columns containing special location information to boolean information
    df["IBA CODE"] = df["IBA CODE"].notnull()
    df["BCR CODE"] = df["BCR CODE"].notnull()
    df["USFWS CODE"] = df["USFWS CODE"].notnull()

    # these columns will help us merge the weather data with the bird data
    df["LATITUDE_ROUNDED"] = np.floor(df["LATITUDE"]) 
    df["LONGITUDE_ROUNDED"] = np.floor(df["LONGITUDE"]) 

    # drop redundunt rows arising from group observations
    individs = df[df["GROUP IDENTIFIER"].isna()].drop(["GROUP IDENTIFIER"], axis = 1)
    grps = df[df["GROUP IDENTIFIER"].notnull()].groupby("GROUP IDENTIFIER").first()
    
    return pd.concat([individs, grps], ignore_index = True)

    

In [22]:
res = processBirdData(df)
print(res.info())
res.to_csv('Data/bird_two.txt', index = False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 828198 entries, 0 to 828197
Data columns (total 14 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   OBSERVATION COUNT          828198 non-null  object 
 1   IBA CODE                   828198 non-null  bool   
 2   BCR CODE                   828198 non-null  bool   
 3   USFWS CODE                 828198 non-null  bool   
 4   LATITUDE                   828198 non-null  float64
 5   LONGITUDE                  828198 non-null  float64
 6   OBSERVATION DATE           828198 non-null  object 
 7   TIME OBSERVATIONS STARTED  813921 non-null  object 
 8   OBSERVATION TYPE           828198 non-null  object 
 9   DURATION MINUTES           814362 non-null  float64
 10  EFFORT DISTANCE KM         465135 non-null  float64
 11  EFFORT AREA HA             4474 non-null    float64
 12  LATITUDE_ROUNDED           828198 non-null  float64
 13  LONGITUDE_ROUNDED          82